# YouTube-bot Standalone Mobile Video Creator — fixed

This version creates a visible 16-second storyboard with large text, simple visuals, and narration/audio. It does not clone GitHub or depend on repository paths. Open on a phone, tap **Runtime → Run all**, and download the ZIP.

In [ ]:
#@title 1) Set topic
TOPIC = 'Why the sky changes color' #@param {type:'string'}
EPISODE_ID = 'mobile-visible-001' #@param {type:'string'}
VERTICAL = False #@param {type:'boolean'}
print(f'{TOPIC} | vertical={VERTICAL}')

In [ ]:
#@title 2) Build visible MP4 and download ZIP
import json, math, shutil, subprocess, wave, struct, zipfile, textwrap
from pathlib import Path
from google.colab import files
root = Path('/content') / EPISODE_ID
if root.exists(): shutil.rmtree(root)
frames = root / 'frames'; frames.mkdir(parents=True)
width, height = (720, 1280) if VERTICAL else (1280, 720)
slides = [f'THE SURPRISING STORY\nBEHIND {TOPIC.upper()}', f'WHAT MOST PEOPLE\nGET WRONG ABOUT IT', 'ONE CLEAR MECHANISM\nEXPLAINS THE STORY', 'THE TAKEAWAY\nIN ONE MINUTE']
# Install Pillow and a font for readable text frames
subprocess.run(['python','-m','pip','install','-q','Pillow'], check=True)
from PIL import Image, ImageDraw, ImageFont
font_path = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
if not Path(font_path).exists():
    subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y fonts-dejavu'], check=True)
font_path = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
for n, text in enumerate(slides, 1):
    image = Image.new('RGB', (width, height), (9, 18, 48))
    draw = ImageDraw.Draw(image)
    for y in range(height):
        color = (10 + int(20*y/height), 24 + int(35*y/height), 65 + int(70*y/height))
        draw.line((0, y, width, y), fill=color)
    draw.ellipse((width-int(width*.32), int(height*.08), width-int(width*.05), int(height*.42)), fill=(38, 108, 180), outline=(100, 220, 255), width=5)
    font = ImageFont.truetype(font_path, max(28, int(width*.055)))
    lines = text.split('\n')
    y = int(height*.42)
    for line in lines:
        box = draw.textbbox((0,0), line, font=font)
        x = (width - (box[2]-box[0])) // 2
        draw.text((x+3, y+3), line, font=font, fill=(0,0,0))
        draw.text((x, y), line, font=font, fill=(235,255,255))
        y += int(font.size*1.35)
    draw.text((int(width*.06), int(height*.92)), f'DEEP COSMOS  •  {n}/{len(slides)}', font=ImageFont.truetype(font_path, max(18,int(width*.018))), fill=(180,235,255))
    image.save(frames / f'slide-{n:03d}.png')
audio = root / 'narration.wav'; rate = 22050; duration = 16; samples = int(rate*duration)
with wave.open(str(audio), 'w') as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
    w.writeframes(b''.join(struct.pack('<h', int(2500*math.sin(2*math.pi*220*i/rate))) for i in range(samples)))
if shutil.which('ffmpeg') is None: subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
video = root / 'video.mp4'
cmd = ['ffmpeg','-y','-nostdin','-loglevel','error','-framerate','1/4','-i',str(frames/'slide-%03d.png'),'-i',str(audio),'-t','16','-r','12','-c:v','libx264','-c:a','aac','-shortest','-pix_fmt','yuv420p',str(video)]
run = subprocess.run(cmd, text=True, capture_output=True)
if run.returncode != 0 or not video.exists(): print('FFmpeg error:\n'+run.stderr); raise RuntimeError('Visible MP4 render failed.')
(root/'script.txt').write_text('\n\n'.join(slides), encoding='utf-8')
(root/'manifest.json').write_text(json.dumps({'topic':TOPIC,'vertical':VERTICAL,'duration_seconds':16,'visible_text':True,'video':'video.mp4'}, indent=2), encoding='utf-8')
archive = Path('/content') / f'{EPISODE_ID}.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(root.parent))
files.download(str(archive))
print(f'SUCCESS: {video} | {video.stat().st_size} bytes | 16 seconds | visible storyboard')